# Demo 13 — Family-wise correction across multiple dependent variables

When you test several outcomes at once, the chance of at least one false positive grows with the number of tests. `options.y_correction` adjusts the per-outcome p-values together (one family per model term) and writes the raw and adjusted values to `MultipleComparisons.xlsx`.

This demo compares six characteristics of the `mtcars` cars (R base; 1974 Motor Trend) between automatic and manual transmission:

    {mpg, hp, wt, qsec, drat, disp} ~ am

Four outcomes differ clearly by transmission and two do not — a realistic family in which correction matters.

## Setup

In [ ]:
import os

import matplotlib.pyplot as plt
from kbstatpy import Kbstat, KbstatOptions

## Options

`y` is a comma-separated list of six outcomes; `x = am` is the transmission factor. `y_correction = 'FDR'` corrects the `am` p-values across the six outcomes (Benjamini-Hochberg). Other choices: `'bonferroni'`, `'holm'` (family-wise error rate — stricter), `'FDR_correlated'` (Benjamini-Yekutieli, for correlated outcomes).

In [ ]:
options = KbstatOptions()
options.in_file      = os.path.join(options.demo_dir, 'data/mtcars.csv')
options.out_dir      = ''   # empty: show results inline only; set a folder to also save them
options.y            = 'mpg, hp, wt, qsec, drat, disp'
options.y_units      = 'mpg, hp, 1000 lb, s, ratio, cu in'
options.x            = 'am'
options.rename       = 'am: 0 -> automatic, 1 -> manual'
options.x_order      = ['automatic', 'manual']
options.y_correction = 'FDR'   # correct the am p-values across the six outcomes

## Run

`run()` fits a model per outcome, then corrects the `am` p-values across the six as one family. The combined table is available as `kb.output.multiple_comparisons` (and written to `MultipleComparisons.xlsx` when `out_dir` is set).

In [ ]:
kb = Kbstat(options)
kb.run()

# Raw p vs FDR-adjusted p for the `am` term across the six outcomes:
kb.output.multiple_comparisons

## Save results

Everything above is shown inline. To also write all tables and figures to disk (including `MultipleComparisons.xlsx`), uncomment the lines below and run this cell — it sets `out_dir` and calls `save()` on the results already computed above (no re-run).

In [ ]:
# options.out_dir = 'results/demo_13_family_correction'
# kb.save()
# kb.download_link()   # remote server: zip the results and click to download

## Interpretation

- `mpg`, `wt`, `drat`, `disp` differ strongly by transmission and stay significant after FDR correction; `hp` and `qsec` do not.
- With six tests, ~0.3 false positives are expected by chance even under the null; the correction guards the family. Compare the `p` and `p_corrected` columns.
- The correction is applied per model term (here just `am`). With covariates it would correct `am`, `Age`, etc. each as its own family.
- This corrects across outcomes within a single run. If your family of tests spans several separate runs (e.g. one per condition), apply the correction at that outer level instead.